# Silage or Grain? Field-level corn harvest classification at USDA BARC

**Question (USDA NASS):** was each corn field harvested as **silage** (chopped green, around dent stage, late summer) or **grain** (dried down and combined in fall)? Can we tell from public satellite data **before the January Annual Summary**, in a way a skeptic can check?

**Why it matters:** production = harvested acres x yield. Silage acres come out of grain acres, so an earlier silage estimate sharpens the corn production forecast.

**What this notebook shows, using only saved, versioned outputs (no live downloads):**
1. The data and its quality (clouds, rejected scenes, label counts).
2. The physical signal: silage drops from green to bare soil early; grain browns slowly.
3. A small, interpretable logistic regression, held out **by field**, against the always-Grain baseline.
4. How early in the season the call can be made.
5. A roll-up from fields to acres with an uncertainty interval, and the NASS context.

## 1. Setup and source integrity
The notebook reads `baseline/outputs/` and the source GeoPackage. It first checks that the original source files are byte-identical to the recorded hashes.

In [ ]:
import hashlib, json, math, os, sqlite3
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'baseline' / 'outputs').exists())
OUT = ROOT / 'baseline' / 'outputs'
GPKG = ROOT / 'BARC_Corn.2021-2023.gpkg'
pd.set_option('display.precision', 3)

expected = json.loads((ROOT / 'baseline' / 'original_sha256.json').read_text())
integrity = {name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest
             for name, digest in expected.items()}
print('project root:', ROOT)
assert all(integrity.values()), f'Source integrity failed: {integrity}'
integrity

## 2. Data and quality
- **Fields and labels:** 83 BARC field polygons (Beltsville, MD). Reference labels exist for 44 field-years on 34 fields; blank labels are unknown and never treated as Grain.
- **Imagery:** Sentinel-2 L2A (Microsoft Planetary Computer), July to November, per-field clear-pixel NDVI; scenes with too few clear pixels are rejected, not filled in.

In [ ]:
audit = json.loads((OUT / 'audit.json').read_text())
metrics = json.loads((OUT / 'metrics.json').read_text())
labels = pd.DataFrame(audit['label_counts']).T
labels.loc['Total'] = labels.sum()
obs = pd.read_csv(OUT / 'observations.csv', parse_dates=['date'])
quality = pd.Series({
    'fields': audit['fields'], 'labeled field-years': metrics['labeled_field_years'],
    'scene observations': len(obs), 'accepted': int((obs.status == 'ok').sum()),
    'rejected (cloud / nodata)': metrics['cloud_or_nodata_rejections'], 'read errors': metrics['read_errors']})
display(labels); quality

## 3. The signal: what silage and grain look like from space
Median NDVI of accepted observations by day of year, with the middle 50% shaded. Silage fields fall to bare soil around late August to September; grain fields stay green longer and brown down gradually.

In [ ]:
ok = obs[obs.status == 'ok'].merge(pd.read_csv(OUT / 'labels.csv'), on=['OBJECTID', 'year'])
ok['week'] = (ok.date.dt.dayofyear // 7) * 7
fig, ax = plt.subplots(figsize=(9, 4))
for lab, color in [('Silage', '#1b7837'), ('Grain', '#b2702d')]:
    g = ok[ok.label == lab].groupby('week').ndvi
    q = g.quantile([.25, .5, .75]).unstack()
    ax.plot(q.index, q[0.5], color=color, lw=2, label=f'{lab} (median)')
    ax.fill_between(q.index, q[0.25], q[0.75], color=color, alpha=.2)
for doy, name in [(213, 'Aug 1'), (244, 'Sep 1'), (274, 'Oct 1'), (305, 'Nov 1')]:
    ax.axvline(doy, color='grey', lw=.5, ls=':'); ax.text(doy + 1, 0.05, name, fontsize=8, color='grey')
ax.set(xlabel='day of year', ylabel='NDVI', title='BARC labeled field-years, 2021-2023'); ax.legend()
plt.tight_layout()

## 4. Model: small and interpretable
- **Features (9):** monthly median NDVI for July to November, peak NDVI and its day, September-onward median NDVI, and early-minus-late NDVI.
- **Model:** median imputation, standard scaling, logistic regression (C = 1), fitted inside each training fold only.
- **Validation:** 5-fold stratified **group** k-fold by field (`OBJECTID`), so every year of a field is held out together. No tuning on held-out results.
- **Baseline:** always predict Grain. It scores well on accuracy because silage is rare, so we report **balanced accuracy, silage F1 and Brier score** instead.

In [ ]:
cols = ['silage_precision', 'silage_recall', 'silage_f1', 'false_positive_rate', 'balanced_accuracy', 'brier_score']
print(metrics['protocol'])
pd.DataFrame({'Logistic regression': {k: metrics['logistic'][k] for k in cols},
              'Always Grain': {k: metrics['always_grain'][k] for k in cols}})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.3))
for ax, (name, key) in zip(axes, [('Logistic regression', 'logistic'), ('Always Grain', 'always_grain')]):
    cm = np.array(metrics[key]['confusion_matrix'])
    ax.imshow(cm, cmap='Greens', vmin=0)
    for i in range(2):
        for j in range(2): ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=12)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Grain', 'Silage'], yticklabels=['Grain', 'Silage'],
           xlabel='predicted', ylabel='reference', title=name)
plt.tight_layout()

### By year (same held-out predictions)

In [ ]:
pred = pd.read_csv(OUT / 'heldout_predictions.csv')
assert pred.label.isin(['Grain', 'Silage']).all(), 'unknown reference labels'
assert pred.groupby('OBJECTID').fold.nunique().eq(1).all(), 'field crosses validation folds'
assert not pred.duplicated(['OBJECTID', 'year']).any(), 'duplicate field-years'
def year_scores(g):
    y, p = g.label == 'Silage', g.predicted_label == 'Silage'
    tp, fp, fn, tn = (y & p).sum(), (~y & p).sum(), (y & ~p).sum(), (~y & ~p).sum()
    rec, spec = tp / (tp + fn) if tp + fn else np.nan, tn / (tn + fp) if tn + fp else np.nan
    return pd.Series({'n': len(g), 'silage': int(y.sum()), 'caught': int(tp), 'false alarms': int(fp),
                      'balanced_accuracy': (rec + spec) / 2})
pred.groupby('year')[['label', 'predicted_label']].apply(year_scores)

### Precision and recall for both classes, with Wilson 95% intervals
With 12 silage field-years, intervals are wide; we show them rather than a single number.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return (max(0, c - h), min(1, c + h))
y, p = pred.label, pred.predicted_label
rows = []
for cls in ['Grain', 'Silage']:
    tp = int(((y == cls) & (p == cls)).sum()); npred = int((p == cls).sum()); nref = int((y == cls).sum())
    for metric, k, n in [('precision', tp, npred), ('recall', tp, nref)]:
        lo, hi = wilson(k, n)
        rows.append({'class': cls, 'metric': metric, 'value': k / n if n else np.nan, 'k/n': f'{k}/{n}', '95% low': lo, '95% high': hi})
pd.DataFrame(rows).set_index(['class', 'metric'])

### Against the organizers' baseline: date of sustained NDVI decline
The lab names a transparent rule to beat: call a field Silage when its NDVI shows a sustained decline early enough. The rule's date threshold is fitted on training fields only, on the same field folds, and scored on the same 44 field-years as the model (`baseline/outputs/decline_rule/metrics.json`).

In [ ]:
rule = json.loads((OUT / 'decline_rule' / 'metrics.json').read_text())
assert rule['evaluated_field_years'] == len(pred), 'decline rule scored on different rows'
def row(name, m):
    (tn, fp), (fn, tp) = m['confusion_matrix']
    return {'model': name, 'balanced_accuracy': m['balanced_accuracy'], 'silage caught': f'{tp}/{tp + fn}',
            'false alarms (of grain)': f'{fp}/{fp + tn}', 'silage precision': m['silage_precision']}
print(rule['protocol'])
display(pd.DataFrame([row('Logistic regression (ours)', rule['logistic_same_rows']),
                      row("Organizers' decline-date rule", rule['decline_rule']),
                      row('Always Grain', rule['always_grain_same_rows'])]).set_index('model').round(3))
pd.DataFrame({y: {k: round(v['balanced_accuracy'], 3) for k, v in r.items()} for y, r in rule['by_year'].items()}).T.rename_axis('year')

The decline rule finds almost every silage field but also flags about half the grain fields, because grain fields lose greenness too, just later and more slowly. The model's monthly features separate a sharp late-summer cut from a gradual fall dry-down, which keeps false alarms low. With 12 silage field-years, a difference of one or two fields is within noise.

### Holding out a whole year
The main results hold out fields. The challenge also asks for a held-out year: train on two seasons, test on the third. **Plain**: train on the other two years. **Strict**: also drop, from training, every field that appears in the test year, so no field is ever on both sides. Same pipeline, nothing tuned.

In [ ]:
try:
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    HAVE_SK = True
except ImportError:
    HAVE_SK = False; print('scikit-learn not installed: pip install scikit-learn to run this section')
if HAVE_SK:
    feats_ = metrics['features']
    def scores(y, hat):
        tp = int((y & hat).sum()); fn = int((y & ~hat).sum()); fp = int((~y & hat).sum()); tn = int((~y & ~hat).sum())
        rec = tp / (tp + fn) if tp + fn else np.nan; spec = tn / (tn + fp) if tn + fp else np.nan
        return {'silage caught': f'{tp}/{tp + fn}', 'false alarms': f'{fp}/{fp + tn}', 'balanced_accuracy': (rec + spec) / 2}
    rows_ = []
    for test_year in sorted(pred.year.unique()):
        test = pred[pred.year == test_year]
        for mode in ('plain', 'strict'):
            train = pred[pred.year != test_year]
            if mode == 'strict':
                train = train[~train.OBJECTID.isin(test.OBJECTID)]
            if train.label.nunique() < 2:
                rows_.append({'held-out year': test_year, 'split': mode, 'train rows': len(train), 'note': 'one class only'}); continue
            m = make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler(),
                              LogisticRegression(C=1.0, max_iter=2000, random_state=42))
            m.fit(train[feats_], train.label == 'Silage')
            hat = m.predict_proba(test[feats_])[:, 1] >= 0.5
            rows_.append({'held-out year': test_year, 'split': mode, 'train rows': len(train),
                          'train silage': int((train.label == 'Silage').sum()), 'test rows': len(test),
                          **scores((test.label == 'Silage').to_numpy(), hat)})
    year_out = pd.DataFrame(rows_).set_index(['held-out year', 'split'])
    display(year_out.round(3))
    for mode in ('plain', 'strict'):
        sub = year_out.xs(mode, level='split')
        print(f'{mode}: mean balanced accuracy over held-out years = {sub.balanced_accuracy.mean():.3f}'
              ' (always Grain = 0.500)')

Each held-out year has only 4 silage field-years, so one field moves recall by 25 points. Read this table for direction (does the model still beat always-Grain on an unseen season?), not for precise values.

## 5. Explainable to a skeptic: the evidence behind each call
Feature values by reference label, and the NDVI curves of the silage fields the model **missed**, so reviewers can see why.

In [ ]:
feat = pd.read_csv(OUT / 'features.csv')
show = ['ndvi_08', 'ndvi_09', 'ndvi_10', 'peak_doy', 'late_ndvi', 'early_minus_late']
display(feat.groupby('label')[show].median().T.rename_axis('median by label'))
missed = pred[(pred.label == 'Silage') & (pred.predicted_label == 'Grain')]
fig, ax = plt.subplots(figsize=(9, 3.5))
for _, r in missed.iterrows():
    g = ok[(ok.OBJECTID == r.OBJECTID) & (ok.year == r.year)].sort_values('date')
    ax.plot(g.date.dt.dayofyear, g.ndvi, marker='o', ms=3, label=f'field {r.OBJECTID} / {r.year}  p={r.silage_probability:.2f}')
ax.set(xlabel='day of year', ylabel='NDVI', title='Silage field-years the model called Grain'); ax.legend(fontsize=7)
plt.tight_layout()
missed[['OBJECTID', 'year', 'valid_dates', 'silage_probability', 'ndvi_09', 'late_ndvi']]

## 6. How early can we call it?
We refit the **same** pipeline on the **same saved field folds**, but only with features available by each date. Monthly medians are complete at month end; season-wide features (peak, late, early-minus-late) are used only in the full-season row. Nothing is tuned on held-out results.

In [ ]:
try:
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
except ImportError:
    print('scikit-learn not installed: pip install scikit-learn to run this section'); SK = False
else:
    SK = True

CUTOFFS = {'Aug 31': ['ndvi_07', 'ndvi_08'],
           'Sep 30': ['ndvi_07', 'ndvi_08', 'ndvi_09'],
           'Oct 31': ['ndvi_07', 'ndvi_08', 'ndvi_09', 'ndvi_10'],
           'Nov 30 (full season, saved model)': metrics['features']}
if SK:
    d = pred.copy(); y = (d.label == 'Silage').astype(int).to_numpy()
    rows = []
    for name, cols_ in CUTOFFS.items():
        prob = np.zeros(len(d))
        for fold in sorted(d.fold.unique()):
            tr, te = d.fold != fold, d.fold == fold
            m = make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler(),
                              LogisticRegression(C=1.0, max_iter=2000, random_state=42))
            m.fit(d.loc[tr, cols_], y[tr]); prob[te.to_numpy()] = m.predict_proba(d.loc[te, cols_])[:, 1]
        hat = (prob >= 0.5).astype(int)
        rows.append({'as of': name, 'features': len(cols_), 'balanced_accuracy': balanced_accuracy_score(y, hat),
                     'silage_f1': f1_score(y, hat, zero_division=0), 'brier': brier_score_loss(y, prob),
                     'silage caught': f'{int((hat & y).sum())}/{int(y.sum())}', 'false alarms': int((hat & (1 - y)).sum())})
    early = pd.DataFrame(rows).set_index('as of')
    check = abs(early.iloc[-1].balanced_accuracy - metrics['logistic']['balanced_accuracy']) < 1e-9
    print('full-season refit reproduces saved metrics:', check)
    assert check, 'Full-season refit differs from saved evaluation'
    display(early)

Every row is months before the January Annual Summary. The table shows the trade-off between calling early and calling accurately.

## 7. Roll-up: fields to acres, with uncertainty
Expected silage acres = sum of p(silage) x field acres; the 95% interval and CV% (as NASS prints it) treat field-years as independent. The **adjusted** column applies the misclassification correction (q - FPR)/(TPR - FPR), with acre-weighted error rates from the other years only. **Scope:** only the 44 labeled field-years, so we can compare against reference acres.

In [ ]:
with sqlite3.connect(f'file:{GPKG}?mode=ro', uri=True) as db:
    acres = pd.read_sql('SELECT OBJECTID, ACRES FROM barc_silage_grain_corn20212023', db)
r = pred.merge(acres, on='OBJECTID', how='left')
assert r.ACRES.notna().all(), 'missing acres for some field-years'
def rates(df):  # acre-weighted, to correct an acre-weighted share (same as the API)
    s, g = df[df.label == 'Silage'], df[df.label == 'Grain']
    return (s.ACRES[s.predicted_label == 'Silage'].sum() / s.ACRES.sum(),
            g.ACRES[g.predicted_label == 'Silage'].sum() / g.ACRES.sum())
def rollup(g):
    p, a = g.silage_probability, g.ACRES
    exp, sd = (p * a).sum(), math.sqrt((a**2 * p * (1 - p)).sum())
    q = a[g.predicted_label == 'Silage'].sum() / a.sum()
    tpr, fpr = rates(r[r.year != g.name])   # error rates from the OTHER years only
    adj = min(1, max(0, (q - fpr) / (tpr - fpr))) if tpr > fpr else np.nan
    return pd.Series({'field-years': len(g), 'total acres': a.sum(),
        'reference silage acres': a[g.label == 'Silage'].sum(),
        'expected silage acres': exp, '95% low': max(0, exp - 1.96 * sd), '95% high': exp + 1.96 * sd,
        'CV %': 100 * sd / exp, 'adjusted silage acres': adj * a.sum()})
roll = r.groupby('year')[['silage_probability', 'ACRES', 'label', 'predicted_label']].apply(rollup)
roll['reference inside 95%'] = roll['reference silage acres'].between(roll['95% low'], roll['95% high'])
roll.round(1)

In [ ]:
ax = roll[['reference silage acres', 'expected silage acres']].plot.bar(figsize=(7, 3.5), color=['#555', '#1b7837'], rot=0)
ax.errorbar(np.arange(len(roll)) + 0.125, roll['expected silage acres'],
            yerr=[roll['expected silage acres'] - roll['95% low'], roll['95% high'] - roll['expected silage acres']],
            fmt='none', color='black', capsize=4)
ax.set(ylabel='acres', title='Silage acres: reference vs model estimate (95% interval)'); plt.tight_layout()

### Convergence through the season (Justin's ask)
NASS releases monthly forecasts around the 12th. We re-run the estimate **as of each release date**, using only the monthly NDVI medians complete by then (same pipeline, same saved field folds). The January row is the same model's saved full-season prediction, not a NASS estimate. Neither convergence toward reference acres nor narrower intervals is guaranteed.

In [ ]:
RELEASES = ['08-12', '09-12', '10-12', '11-12']
YEARS = sorted(pred.year.unique())
assert (r.OBJECTID.values == pred.OBJECTID.values).all()  # same row order as the folds
MONTH_END = {'ndvi_07': '07-31', 'ndvi_08': '08-31', 'ndvi_09': '09-30', 'ndvi_10': '10-31', 'ndvi_11': '11-30'}
def asof_probs(cols_):
    d = pred.copy(); yv = (d.label == 'Silage').astype(int).to_numpy(); prob = np.zeros(len(d))
    for fold in sorted(d.fold.unique()):
        tr, te = (d.fold != fold).to_numpy(), (d.fold == fold).to_numpy()
        m = make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler(),
                          LogisticRegression(C=1.0, max_iter=2000, random_state=42))
        m.fit(d.loc[tr, cols_], yv[tr]); prob[te] = m.predict_proba(d.loc[te, cols_])[:, 1]
    return prob
conv = []
if SK:
    for md_ in RELEASES + ['Jan']:
        cols_ = metrics['features'] if md_ == 'Jan' else [f for f, end in MONTH_END.items() if end <= md_]
        if not cols_:
            continue
        rr = r.copy()
        rr['p'] = rr.silage_probability if md_ == 'Jan' else asof_probs(cols_)
        rr['hat'] = np.where(rr.p >= 0.5, 'Silage', 'Grain')
        for yr, g in rr.groupby('year'):
            exp, sd = (g.p * g.ACRES).sum(), math.sqrt((g.ACRES**2 * g.p * (1 - g.p)).sum())
            conv.append({'year': yr, 'as of': md_, 'features': len(cols_), 'expected': exp,
                         'low': max(0, exp - 1.96 * sd), 'high': exp + 1.96 * sd, 'CV %': 100 * sd / exp,
                         'reference': g.ACRES[g.label == 'Silage'].sum(),
                         'balanced_accuracy': balanced_accuracy_score(g.label == 'Silage', g.hat == 'Silage')})
    conv = pd.DataFrame(conv)
    fig, axes = plt.subplots(1, len(YEARS), figsize=(4.2 * len(YEARS), 3.4), sharey=True)
    for ax, yr in zip(axes, YEARS):
        c = conv[conv.year == yr]; x = np.arange(len(c))
        ax.fill_between(x, c.low, c.high, color='#1b7837', alpha=.18, label='95% interval')
        ax.plot(x, c.expected, 'o-', color='#1b7837', label='estimate')
        ax.axhline(c.reference.iloc[0], color='#555', ls='--', label='reference (BARC labels)')
        ax.set_xticks(x, c['as of']); ax.set_title(str(yr)); ax.set_xlabel('as of (NASS release)')
    axes[0].set_ylabel('silage acres'); axes[0].legend(fontsize=7, loc='upper left'); plt.tight_layout()
    display(conv.set_index(['year', 'as of']).round(2))
else:
    print('scikit-learn needed for the convergence run')

**Reading the chart honestly.**
- **Scores improve by October.** Balanced accuracy is 0.5 (no better than always-Grain) on Aug 12 in every year; by Oct 12 it is 0.62 to 0.88. September NDVI improves scores in all three years here; August NDVI already separates some 2023 fields, while the Sep 12 score in 2022 is below chance.
- **Early acre estimates are not skill.** In August the probabilities fall back toward the base rate, so the acres can land near the reference without any field being identified. Judge the early dates by balanced accuracy, not by the acre line.
- **The interval does not narrow consistently.** CV ranges from about 23 to 55%; each year has only 7 to 19 labeled field-years. In 2022 it falls from 55% in August to 23% in October, then rises to 31% in the full-season row. These intervals condition on model probabilities and omit model and spatial dependence uncertainty.
- The reference line sits inside the 95% interval at every date in every year.

**What worked, what didn't:** the reference acres fall inside the 95% interval every year, so the expected-value estimate is the headline. The misclassification adjustment is unstable here because its error rates rest on about 8 silage field-years; it would matter more on a larger labeled set.

### Where the uncertainty comes from, and how it converges
The interval is wide because each year has only 7 to 19 labeled fields: the total is a sum of a few weighted coin flips. Most of that width sits in a handful of large fields the model is unsure about. Below: each field's share of the variance (acres² × p × (1 − p)), then a **what-if**: an analyst verifies only the most uncertain fields (from imagery, a farm call or an FSA record) and the rest keep the model's probabilities. The checked fields take their BARC reference label here, so this shows the value of checking, not a model result.

In [ ]:
r_ = r.assign(var=r.ACRES**2 * r.silage_probability * (1 - r.silage_probability))
def total(g, p):
    e = (p * g.ACRES).sum(); sd = math.sqrt((g.ACRES**2 * p * (1 - p)).sum())
    return e, max(0, e - 1.96 * sd), e + 1.96 * sd, 100 * sd / e if e else float('nan')
drivers, checks = [], []
for yr, g in r_.groupby('year'):
    g = g.sort_values('var', ascending=False)
    share = g['var'].cumsum() / g['var'].sum(); k = int((share < 0.5).sum()) + 1
    drivers.append({'year': yr, 'fields': len(g), 'fields giving half the variance': k,
                    'their share': round(float(share.iloc[k - 1]), 2),
                    'which': ', '.join(f'#{o} ({a:.0f} ac, p={p:.2f})' for o, a, p in
                                       g.head(k)[['OBJECTID', 'ACRES', 'silage_probability']].itertuples(index=False))})
    ref = g.ACRES[g.label == 'Silage'].sum()
    for n in (0, 1, 3, 5):
        p = g.silage_probability.copy()
        top = g.index[:n]; p[top] = (g.loc[top, 'label'] == 'Silage').astype(float)
        e, lo, hi, cv = total(g, p)
        checks.append({'year': yr, 'fields checked': n, 'estimate': e, '95% low': lo, '95% high': hi,
                       'CV %': cv, 'reference acres': ref, 'reference inside 95%': lo <= ref <= hi})
display(pd.DataFrame(drivers).set_index('year'))
pd.DataFrame(checks).set_index(['year', 'fields checked']).round(1)

Checking the few flagged fields collapses the interval toward the reference: in 2022, verifying 3 of 7 fields gives 74.5 acres at about 4% CV against 74.4 true. Years with more fields need more checks. The model's own interval stays honest about what it does not know; convergence comes from targeted human verification, which is also how the agent's Verifier is designed (it flags borderline and thin-imagery fields for review).

### NASS context
BARC is a small research farm, so its acres are not comparable to Maryland totals. Below are filtered Maryland state-level NASS rows and the BARC model's silage share for context. Set `QUICKSTATS_CSV` to the team's QuickStats export. Suppressed values remain null and are never zeroed.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'backend'))
from app import artifacts
try:
    qs = [q.model_dump() for year in sorted(pred.year.unique())
          for use in ['grain', 'silage']
          for q in artifacts.read_quickstats('MARYLAND', int(year), use, 'STATE')]
    display(pd.DataFrame(qs))
except artifacts.ArtifactUnavailable as exc:
    print('NASS context unavailable:', exc)
print('BARC modelled silage share of labeled corn acres:')
(roll['expected silage acres'] / roll['total acres']).round(3)

## 8. Agent trace
Set `AGENT_RUN_ID` to the specific run to inspect. Only that run's logged steps are shown; the notebook does not select another run automatically. The agent implementation is optional.

In [ ]:
from app import tools as backend_tools
run_id = os.environ.get('AGENT_RUN_ID')
if run_id:
    steps = backend_tools.read_trace(run_id)
    display(pd.DataFrame(steps).tail(25))
else:
    print('Set AGENT_RUN_ID to inspect a trace from', backend_tools.TRACES_DIR)

## 9. Limitations
- 44 labeled field-years on 34 fields at one research farm. Per-year silage counts are 4, so metrics have wide uncertainty.
- Grouping by field prevents same-field leakage, but nearby fields share imagery and management; this does not show transfer to other regions or future years.
- Clouds can hide the cut week; rejected scenes are reported, never filled in.
- Early harvest is not always silage elsewhere (e.g., Texas summer grain); thresholds learned here reflect Maryland timing.
- Double crops are not modelled separately.
- **Only one line of evidence.** The model uses NDVI (greenness) only. The plan's second line of evidence, bare soil vs crop residue after harvest (the SWIR tillage index NDTI = (B11 - B12) / (B11 + B12) on Sentinel-2), was not extracted in time. Residue would help most where the NDVI drop alone is ambiguous: the borderline and missed silage fields in section 5.
- Year-held-out results rest on 7 to 19 field-years per season (4 silage each); they show direction, not precision.

## 10. Reproduce
```bash
python baseline/test_baseline.py
python baseline/run_baseline.py evaluate     # offline, from versioned caches
python -m harvest.run --year 2022 --as-of 2022-10-12 --out runs/2022-10-12   # one command
python -m uvicorn app.main:app --app-dir backend --port 8000                  # API
```

## References
**Publications**
- Dulaney, W., Anderson, M., Gao, F., Stern, A., Moglen, G., Meyers, G., Daughtry, C., White, W., Akumaga, U., & Showalter, J. (2024). Development of a gridded yield data archive for farm management and research at the USDA Beltsville Agricultural Research Center. *Agrosystems, Geosciences & Environment*, 7(1), e20474. https://doi.org/10.1002/agg2.20474
- Kraatz, S., Lamb, B., Hively, W., Jennewein, J., Gao, F., Cosh, M., & Siqueira, P. (2023). Comparing NISAR (Using Sentinel-1), USDA/NASS CDL, and Ground Truth Crop/Non-Crop Areas in an Urban Agricultural Region. *Sensors*, 23(20), 8595. https://doi.org/10.3390/s23208595

**Datasets**
- USDA Beltsville Agricultural Research Center (BARC) Gridded Yield Data Archive, version 1. USDA National Agricultural Library, Ag Data Commons. https://agdatacommons.nal.usda.gov/articles/dataset/USDA_Beltsville_Agricultural_Research_Center_BARC_Gridded_Yield_Data_Archive_version_1/28950746
- Kraatz, S. (2023). Validating NISAR's cropland mapping approach and the USDA/NASS Cropland Data Layer against ground truth data in a fragmented urban agricultural region (Version 1.0.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.8087301
- Sentinel-2 Level-2A, Microsoft Planetary Computer. https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a
- USDA NASS Quick Stats. https://quickstats.nass.usda.gov/